# Attrito viscoso: sferetta d'acciaio nella glicerina — classe 3R

Simulazione con il **metodo passo per passo** (numpy e matplotlib). VPython non funziona in Google Colab: per l'animazione 3D si usa lo script su https://glowscript.org.

Forze sulla sferetta (asse $y$ verso il basso): peso $mg$, spinta di Archimede $\rho_f V g$ verso l'alto, attrito viscoso $b\,v$ verso l'alto, con $b = 6\pi\eta r$ (legge di Stokes).
$$m\,a = mg - \rho_f V g - b\,v$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# dati (SI)
g = 9.81          # m/s^2
r = 2.0e-3        # raggio della sferetta (m)
rho_s = 7800      # acciaio (kg/m^3)
rho_f = 1260      # glicerina (kg/m^3)
eta = 1.4         # viscosità della glicerina (Pa·s)

V = 4/3*np.pi*r**3
m = rho_s*V
b = 6*np.pi*eta*r

a0 = g*(1 - rho_f/rho_s)
vL = (m - rho_f*V)*g/b
tau = m/b
Re = rho_f*vL*2*r/eta
print(f"m = {m*1000:.3f} g,  b = {b:.4f} kg/s")
print(f"a0 = {a0:.2f} m/s^2,  vL = {vL*100:.2f} cm/s,  tau = {tau*1000:.2f} ms,  Re = {Re:.2f}")

## Metodo passo per passo
A ogni passo: calcolo l'accelerazione con la velocità attuale, poi aggiorno la velocità: $v_{n+1} = v_n + a_n\,\Delta t$.

In [ ]:
def passo_per_passo(dt, t_fine=0.03):
    n = int(round(t_fine/dt))
    t = np.zeros(n+1); v = np.zeros(n+1); a = np.zeros(n+1)
    for i in range(n):
        a[i] = (m*g - rho_f*V*g - b*v[i])/m
        v[i+1] = v[i] + a[i]*dt
        t[i+1] = t[i] + dt
    a[n] = (m*g - rho_f*V*g - b*v[n])/m
    return t, v, a

# tabella con dt = 1 ms (la stessa della sintesi)
t1, v1, a1 = passo_per_passo(1e-3, 0.007)
print(" n  t(ms)  v(cm/s)  a(m/s^2)")
for i in range(len(t1)):
    print(f"{i:2d}  {t1[i]*1000:4.0f}   {v1[i]*100:6.2f}   {a1[i]:6.2f}")

In [ ]:
t, v, a = passo_per_passo(1e-4)        # dt = 0,1 ms
t_esatto = np.linspace(0, 0.03, 300)
v_esatto = vL*(1 - np.exp(-t_esatto/tau))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t*1000, v*100, label="passo per passo (dt = 0,1 ms)")
ax[0].plot(t_esatto*1000, v_esatto*100, "--", label="v_L(1 - e^(-t/τ))")
ax[0].plot(t1*1000, v1*100, "o", label="dt = 1 ms")
ax[0].axhline(vL*100, color="gray", lw=0.8)
ax[0].set_xlabel("t (ms)"); ax[0].set_ylabel("v (cm/s)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(t*1000, a)
ax[1].axhline(g, color="red", ls="--", label="g (caduta nel vuoto)")
ax[1].set_xlabel("t (ms)"); ax[1].set_ylabel("a (m/s²)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

i63 = np.argmax(v >= 0.632*vL)
print(f"v al 63% di vL dopo {t[i63]*1000:.2f} ms (tau teorico = {tau*1000:.2f} ms)")
print(f"v dopo 30 ms = {v[-1]*100:.3f} cm/s (vL = {vL*100:.3f} cm/s)")

## Esperimento: $v_L$ in funzione del raggio
Se $v_L \propto r^2$, il grafico di $v_L$ in funzione di $r^2$ è una retta per l'origine.

In [ ]:
raggi = np.array([1.0, 1.5, 2.0, 2.5])*1e-3
vL_r = 2*raggi**2*(rho_s - rho_f)*g/(9*eta)
for rr, vv in zip(raggi, vL_r):
    print(f"r = {rr*1000:.1f} mm   r^2 = {(rr*1000)**2:.2f} mm^2   vL = {vv*100:.2f} cm/s   Re = {rho_f*vv*2*rr/eta:.2f}")
plt.plot((raggi*1000)**2, vL_r*100, "o-")
plt.xlabel("r² (mm²)"); plt.ylabel("v_L (cm/s)"); plt.grid(alpha=0.3); plt.show()

## Sfida: la stessa sferetta nell'acqua
Con la formula di Stokes viene una velocità limite enorme. Il modello è ancora valido? Controlla il numero di Reynolds e confronta con la resistenza quadratica $\tfrac12 C\rho_f A v^2$ ($C = 0{,}4$).

In [ ]:
rho_acqua, eta_acqua, C = 1000, 1.0e-3, 0.4
vL_stokes = 2*r**2*(rho_s - rho_acqua)*g/(9*eta_acqua)
Re_stokes = rho_acqua*vL_stokes*2*r/eta_acqua
A = np.pi*r**2
vL_quad = np.sqrt((m - rho_acqua*V)*g/(0.5*C*rho_acqua*A))
Re_quad = rho_acqua*vL_quad*2*r/eta_acqua
print(f"Stokes:     vL = {vL_stokes:.1f} m/s,  Re = {Re_stokes:.1e}  -> Stokes NON vale")
print(f"quadratica: vL = {vL_quad:.2f} m/s,  Re = {Re_quad:.0f}  -> regime turbolento, modello coerente")